# Проміжний урок. Розрізи складальних одиниць
## Монтажний вузол: просторова модель, проєкція та складальний розріз

**Мета:** навчитися читати та виконувати розріз складальної одиниці, розрізняти зображення деталей, що перетинаються січною площиною, і стандартних кріпильних елементів.

У навчальному вузлі є:
- монтажна основа;
- L-подібний кронштейн;
- два болти;
- шайби;
- гайки.

**Принцип побудови:** січна площина фіксована та проходить через осі обох кріпильних комплектів. Тому болти, шайби й гайки обов'язково присутні на зображенні розрізу. Для поздовжнього розрізу стандартні кріпильні деталі показані без штрихування.

Модель є навчальною геометричною візуалізацією, а не повним генератором креслень за стандартами.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Polygon, Circle
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

plt.rcParams['figure.figsize'] = (15, 7)

# Основні розміри, мм
L = 160          # довжина основи
W = 90           # ширина вузла
Tb = 10          # товщина основи
T = 8            # товщина полиць кронштейна
H = 75           # висота вертикальної полиці
x0 = 22          # координата вертикальної полиці
bolt_x = [65, 125]
bolt_r = 4
washer_r = 8
washer_t = 2
head_h = 5
nut_h = 7

# Січна площина проходить через осі болтів
section_z = W / 2


## 1. Просторова модель складальної одиниці

Орієнтацію 3D-вигляду повернуто на 90° відносно попередньої версії, щоб просторове зображення краще узгоджувалося з проєкцією та розрізом.

Кріпильний комплект змодельовано послідовно: головка болта під основою, стрижень болта, шайба та гайка над горизонтальною полицею кронштейна.

In [ ]:
def add_box(ax, x1,x2,y1,y2,z1,z2, color, alpha=0.96):
    v=[(x1,y1,z1),(x2,y1,z1),(x2,y2,z1),(x1,y2,z1),
       (x1,y1,z2),(x2,y1,z2),(x2,y2,z2),(x1,y2,z2)]
    faces=[[v[0],v[1],v[2],v[3]],[v[4],v[5],v[6],v[7]],
           [v[0],v[1],v[5],v[4]],[v[1],v[2],v[6],v[5]],
           [v[2],v[3],v[7],v[6]],[v[3],v[0],v[4],v[7]]]
    ax.add_collection3d(Poly3DCollection(
        faces,facecolors=color,edgecolors="#263746",linewidths=.7,alpha=alpha))

def add_cylinder_y(ax,x,z,y1,y2,r,color):
    a=np.linspace(0,2*np.pi,48)
    A,Y=np.meshgrid(a,[y1,y2])
    ax.plot_surface(x+r*np.cos(A),Y,z+r*np.sin(A),
                    color=color,linewidth=0,alpha=.98,shade=True)
    for y in [y1,y2]:
        ax.plot(x+r*np.cos(a),np.full_like(a,y),z+r*np.sin(a),
                color="#263746",lw=.55)

def add_hex_prism_y(ax,x,z,y1,y2,r,color):
    a=np.linspace(0,2*np.pi,7)[:-1]+np.pi/6
    p=[(x+r*np.cos(t),z+r*np.sin(t)) for t in a]
    lo=[(xx,y1,zz) for xx,zz in p]
    hi=[(xx,y2,zz) for xx,zz in p]
    faces=[lo,hi]+[[lo[i],lo[(i+1)%6],hi[(i+1)%6],hi[i]] for i in range(6)]
    ax.add_collection3d(Poly3DCollection(
        faces,facecolors=color,edgecolors="#263746",linewidths=.65,alpha=.99))

def draw_3d_assembly(ax, show_section_plane=True):
    # Основа та L-кронштейн
    add_box(ax,0,L,0,Tb,0,W,"#aabac7")
    add_box(ax,x0,L-12,Tb,Tb+T,0,W,"#d9ad68")
    add_box(ax,x0,x0+T,Tb+T,Tb+T+H,0,W,"#d9ad68")

    # Два повні кріпильні комплекти
    for x in bolt_x:
        # Стрижень болта проходить крізь основу та полицю
        add_cylinder_y(ax,x,section_z,-head_h,Tb+T+washer_t+nut_h+2,
                       bolt_r,"#687987")
        # Головка болта під основою
        add_hex_prism_y(ax,x,section_z,-head_h,0,bolt_r*1.8,"#8d969e")
        # Шайба над горизонтальною полицею
        add_cylinder_y(ax,x,section_z,Tb+T,Tb+T+washer_t,
                       washer_r,"#c4cbd1")
        # Гайка над шайбою
        add_hex_prism_y(ax,x,section_z,Tb+T+washer_t,
                        Tb+T+washer_t+nut_h,bolt_r*1.8,"#9da5ac")

    if show_section_plane:
        xx,yy=np.meshgrid([0,L],[-head_h,Tb+T+H+5])
        zz=np.full_like(xx,section_z)
        ax.plot_surface(xx,yy,zz,color="#e9b64f",alpha=.22,shade=False)

    ax.set_xlim(0,L)
    ax.set_ylim(-head_h-3,Tb+T+H+6)
    ax.set_zlim(0,W)
    ax.set_xlabel("X — довжина")
    ax.set_ylabel("Y — висота")
    ax.set_zlabel("Z — ширина")
    ax.set_box_aspect((L,Tb+T+H+head_h,W))
    ax.view_init(elev=22,azim=35)  # поворот камери на 90° відносно попереднього azim=-55
    ax.set_title("Просторова складальна одиниця")


## 2. Проєкція та поздовжній складальний розріз

Січна площина проходить по середині ширини вузла (`Z = W/2`) і через осі обох болтів.

На розрізі:
- основу та кронштейн штрихуємо як перетнуті деталі;
- болти, їхні головки, шайби та гайки показуємо без штрихування;
- кріпильні елементи залишаються видимими в складанні, а не зникають із зображення;
- отвори в основі та полиці умовно відображаємо проміжками в штрихуванні навколо стрижнів болтів.

In [ ]:
def draw_hatched_segment(ax,x1,x2,y1,y2,hatch="////",face="#e4e8eb"):
    if x2 <= x1:
        return
    ax.add_patch(Rectangle((x1,y1),x2-x1,y2-y1,
                           facecolor=face,edgecolor="black",
                           hatch=hatch,linewidth=1.1))

def draw_section(ax):
    # Основа: дві ділянки штрихування з отворами навколо болтів
    r=bolt_r
    for i,x in enumerate(bolt_x):
        left = 0 if i==0 else bolt_x[i-1]+r
        draw_hatched_segment(ax,left,x-r,0,Tb,"////")
    draw_hatched_segment(ax,bolt_x[-1]+r,L,0,Tb,"////")

    # Горизонтальна полиця кронштейна: теж розділена отворами
    x_left=x0
    x_right=L-12
    for i,x in enumerate(bolt_x):
        left = x_left if i==0 else bolt_x[i-1]+r
        draw_hatched_segment(ax,left,x-r,Tb,Tb+T,"\\\\")
    draw_hatched_segment(ax,bolt_x[-1]+r,x_right,Tb,Tb+T,"\\\\")

    # Вертикальна полиця
    ax.add_patch(Rectangle((x0,Tb+T),T,H,
                           facecolor="#e4e8eb",edgecolor="black",
                           hatch="////",linewidth=1.1))

    # Контури основних деталей
    ax.plot([0,L,L,0,0],[0,0,Tb,Tb,0],color="black",lw=1.2)
    ax.plot([x0,x_right,x_right,x0+T,x0+T,x0,x0],
            [Tb,Tb,Tb+T,Tb+T,Tb+T+H,Tb+T+H,Tb],
            color="black",lw=1.2)

    # Кріпильні комплекти: усі елементи присутні, без штрихування
    for x in bolt_x:
        # стрижень болта
        ax.add_patch(Rectangle((x-r,-head_h),2*r,
                               Tb+T+washer_t+nut_h+head_h,
                               facecolor="white",edgecolor="black",lw=1.15,zorder=5))
        # головка болта під основою
        ax.add_patch(Rectangle((x-r*1.8,-head_h),3.6*r,head_h,
                               facecolor="#f5f5f5",edgecolor="black",lw=1.15,zorder=6))
        # шайба над полицею
        ax.add_patch(Rectangle((x-washer_r,Tb+T),2*washer_r,washer_t,
                               facecolor="white",edgecolor="black",lw=1.15,zorder=7))
        # гайка над шайбою
        ax.add_patch(Rectangle((x-r*1.8,Tb+T+washer_t),3.6*r,nut_h,
                               facecolor="#f5f5f5",edgecolor="black",lw=1.15,zorder=7))
        # вісь болта
        ax.plot([x,x],[-head_h-2,Tb+T+washer_t+nut_h+3],
                color="black",lw=.7,linestyle=(0,(5,3,1,3)),zorder=8)

    ax.set_xlim(-12,L+12)
    ax.set_ylim(-head_h-8,Tb+T+H+12)
    ax.set_aspect("equal",adjustable="box")
    ax.set_xlabel("X — довжина, мм")
    ax.set_ylabel("Y — висота, мм")
    ax.set_title("Поздовжній складальний розріз (Z = W/2)")
    ax.grid(True,alpha=.2)

fig=plt.figure(figsize=(16,7))
ax3=fig.add_subplot(121,projection="3d")
ax2=fig.add_subplot(122)
draw_3d_assembly(ax3,True)
draw_section(ax2)
plt.tight_layout()
plt.show()


## 3. Як читати отримане зображення

Порівняйте ліву та праву частини рисунка:
- зліва — просторова складальна одиниця та положення січної площини;
- справа — відповідний поздовжній складальний розріз;
- площина проходить через середини кріпильних комплектів, тому всі болти, шайби та гайки показані в розрізі;
- штрихування позначає матеріал основи та кронштейна, який перетинає площина;
- кріпильні деталі показані без штрихування.

**Зверніть увагу:** у реальному складальному кресленні напрямок і крок штрихування суміжних деталей добирають так, щоб розрізняти сусідні компоненти. Тут штрихування навмисно спрощене для навчального пояснення.

## 4. Практичні завдання

1. Знайдіть на 3D-моделі січну площину та поясніть, чому вона проходить через обидва кріпильні комплекти.
2. Визначте, які компоненти перетинаються площиною та мають бути заштриховані.
3. Поясніть, чому болти, шайби й гайки показані без штрихування.
4. Зіставте кожен елемент на 3D-моделі з його зображенням на розрізі.
5. Запропонуйте інше положення січної площини, яке дозволило б показати внутрішню будову вузла ще наочніше. Обґрунтуйте вибір.

## 5. Контрольні запитання

1. Для чого виконують розріз складальної одиниці?
2. Що показує розріз, крім фігури, утвореної січною площиною?
3. Які деталі в цьому прикладі штрихуються?
4. Чому стандартні кріпильні деталі в поздовжньому розрізі зазвичай не штрихують, якщо площина проходить уздовж їхніх осей?
5. Чим розріз відрізняється від перерізу?

**Очікуваний результат:** студент уміє зіставити просторову складальну одиницю з її проєкцією та розрізом, розпізнає компоненти й пояснює умовності їхнього зображення.